# cAPTure: graph logit tail and threshold-scope audit

This audit investigates why ST-GNN has strong within-scenario ranking but very low recall at the frozen global false-alert budget. It uses the existing development OOF logits and prepared packet metadata. Fold-local and scenario-local thresholds are in-sample diagnostics only; the frozen global threshold remains the operational result. No GPU, checkpoint, or training is required.


## 1. Prepare Colab and mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only", "origin",
                    REPOSITORY_BRANCH], cwd=PROJECT_ROOT, check=True)
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
               check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_logit_tail_audit import (
    run_logit_tail_audit, validate_logit_tail_audit,
)
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CPU runtime is sufficient.")

Mounted at /content/drive
Repository commit: 5be046455e7d8d7f2f2fb52f4742f974057669cf
CPU runtime is sufficient.


## 2. Bind immutable development inputs

In [2]:
RESCORING_RUN_ID = "20261001_graph_oof_logit_rescoring_v1"
COMPARISON_RUN_ID = "20261001_graph_logit_comparison_v1"
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
AUDIT_RUN_ID = "20261002_graph_logit_tail_audit_v1"

AUDIT_ARGUMENTS = {
    "config_path": PROJECT_ROOT / "configs/capture_graph_logit_tail_audit_v1.yaml",
    "comparison_dir": DRIVE_ROOT / "graph_logit_comparison_runs" / COMPARISON_RUN_ID,
    "rescoring_dir": DRIVE_ROOT / "graph_oof_rescoring_runs" / RESCORING_RUN_ID,
    "prepared_dir": DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID,
    "output_dir": DRIVE_ROOT / "graph_logit_tail_audit_runs" / AUDIT_RUN_ID,
}
for name, path in AUDIT_ARGUMENTS.items():
    if name != "output_dir" and not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
print("Comparison input:", AUDIT_ARGUMENTS["comparison_dir"])
print("Rescoring input:", AUDIT_ARGUMENTS["rescoring_dir"])
print("Audit output:", AUDIT_ARGUMENTS["output_dir"])

Comparison input: /content/drive/MyDrive/capture_gate0/graph_logit_comparison_runs/20261001_graph_logit_comparison_v1
Rescoring input: /content/drive/MyDrive/capture_gate0/graph_oof_rescoring_runs/20261001_graph_oof_logit_rescoring_v1
Audit output: /content/drive/MyDrive/capture_gate0/graph_logit_tail_audit_runs/20261002_graph_logit_tail_audit_v1


## 3. Run or validate the audit

Review the paths above and set `RUN_AUDIT=True`. The audit reads the existing OOF files and saves a new immutable report in Drive. Reopening a completed run validates its hashes.

In [3]:
RUN_AUDIT = True

if AUDIT_ARGUMENTS["output_dir"].exists():
    audit_report = validate_logit_tail_audit(**AUDIT_ARGUMENTS)
elif RUN_AUDIT:
    audit_report = run_logit_tail_audit(**AUDIT_ARGUMENTS)
else:
    audit_report = None
    print("Audit disabled. Set RUN_AUDIT=True to create the report.")

Auditing score tails for Edge MLP...
Auditing score tails for EdgeGRU...
Auditing score tails for StaticGNN...
Auditing score tails for ST-GNN...


## 4. Inspect score scales and false-alert tails

In [4]:
if audit_report is None:
    print("Run section 3 before reviewing results.")
else:
    print("Audit status:", audit_report["status"])
    display(pd.DataFrame(audit_report["fold_scale_summary"]))
    display(pd.DataFrame(audit_report["packet_logit_tails"]))
    display(pd.DataFrame(audit_report["window_logit_tails"]))
    display(pd.DataFrame(audit_report["threshold_scope_metrics"]))
    display(pd.DataFrame(audit_report["st_gnn_top_window_metadata"]))
    display(Markdown((AUDIT_ARGUMENTS["output_dir"] / "review.md").read_text(encoding="utf-8")))

Audit status: exploratory_logit_tail_audit_complete


,model,fold,frozen_global_threshold,fold_local_threshold,maximum_attack_packet_logit,benign_window_logit_q99,benign_window_logit_q999,maximum_benign_window_logit,attack_window_logit_q99,global_attack_packets_above_threshold,global_fold_mean_false_alert_windows_per_hour,local_fold_mean_false_alert_windows_per_hour,global_fold_mean_packet_recall,local_fold_mean_packet_recall
0,edge_mlp,A,10.284599,5.959848,6.101065,5.222249,5.959848,6.073847,6.095416,0,0.000000,0.489692,0.000000,0.002805
1,edge_mlp,B,10.284599,10.284599,14.216803,3.446996,10.284599,10.284599,11.973738,642788,0.000000,0.000000,0.546343,0.546343
2,edge_gru,A,24.407442,24.407442,39.638981,22.642641,24.725087,25.153351,33.799892,4518685,0.984721,0.984721,0.527397,0.527397
3,edge_gru,B,24.407442,-3.320819,17.774080,-3.944568,-3.320818,-1.344180,13.916284,0,0.000000,0.932941,0.000000,0.759193
4,static_gnn,A,7.059270,7.059270,18.037746,2.447683,7.712916,14.641726,17.281052,5542467,0.968959,0.968959,0.769804,0.769804
5,static_gnn,B,7.059270,6.898990,13.778070,3.520528,9.335387,9.514657,9.256675,757590,0.946445,0.946445,0.640654,0.645275
6,st_gnn,A,17.958267,17.958267,18.878252,17.352619,18.010876,18.165426,17.981376,531,0.966029,0.966029,0.000155,0.000155
7,st_gnn,B,17.958267,6.377722,13.378479,2.228587,6.653571,8.758240,10.769270,0,0.000000,0.946445,0.000000,0.808617


,model,fold,scenario,class,packets,minimum_raw_logit,q50,q90,q99,q999,q9999,maximum_raw_logit
0,edge_mlp,A,train_dollar_char,benign,1705003,-7.783275,-7.700543,-5.857592,-4.068290,1.775407,6.029623,6.073847
1,edge_mlp,A,train_dollar_char,attack,1177552,-7.768583,1.609932,5.686985,5.894906,5.958747,6.080459,6.101065
2,edge_mlp,A,train_slash_char,benign,1705003,-7.783275,-7.700643,-5.858232,-4.068290,1.837559,5.936603,6.073847
3,edge_mlp,A,train_slash_char,attack,4720513,-7.403652,4.915912,5.763533,5.915007,5.960383,6.073534,6.101064
4,edge_mlp,A,train_sub_exf,benign,1705003,-7.783275,-7.700497,-5.857379,-4.067767,1.787430,4.459794,6.073847
5,edge_mlp,A,train_sub_exf,attack,520804,-7.281903,4.899607,5.737547,5.936769,6.080041,6.098623,6.101065
6,edge_mlp,B,train_empty_conn,benign,746806,-16.669395,-16.288145,-6.207626,0.402124,3.263810,7.163215,10.284599
7,edge_mlp,B,train_empty_conn,attack,428973,-12.042600,10.696805,13.100623,14.003960,14.129722,14.205613,14.216803
8,edge_mlp,B,train_qos_mid,benign,746806,-16.669395,-16.288145,-6.266271,0.448068,3.313615,6.005087,10.284599
9,edge_mlp,B,train_qos_mid,attack,752911,-16.616892,10.724617,12.128014,14.003960,14.131926,14.208163,14.216803


,model,fold,scenario,class,observed_windows,empty_windows,minimum_raw_logit,q50,q90,q99,q999,q9999,maximum_raw_logit
0,edge_mlp,A,train_dollar_char,benign,5569,244,-6.883310,-4.735791,-4.068290,5.222249,5.959848,6.073847,6.073847
1,edge_mlp,A,train_dollar_char,attack_containing,3547,0,-4.735792,2.009120,5.883858,6.073847,6.101064,6.101065,6.101065
2,edge_mlp,A,train_slash_char,benign,6386,244,-6.883310,-4.735791,-4.068290,5.222248,5.959848,6.073847,6.073847
3,edge_mlp,A,train_slash_char,attack_containing,2730,0,-5.320655,5.901056,6.073847,6.073847,6.095416,6.099523,6.101064
4,edge_mlp,A,train_sub_exf,benign,6431,245,-6.883310,-4.735791,-4.068290,5.222249,5.959848,6.073847,6.073847
5,edge_mlp,A,train_sub_exf,attack_containing,2684,0,-5.320655,-4.068290,5.959848,6.101064,6.101064,6.101065,6.101065
6,edge_mlp,B,train_empty_conn,benign,4068,0,-9.634024,-5.405030,-3.887522,3.446996,10.284599,10.284599,10.284599
7,edge_mlp,B,train_empty_conn,attack_containing,5292,0,-8.057443,2.341809,10.284599,12.084982,14.210255,14.216803,14.216803
8,edge_mlp,B,train_qos_mid,benign,4404,0,-9.634024,-5.405030,-3.887522,3.424742,10.284599,10.284599,10.284599
9,edge_mlp,B,train_qos_mid,attack_containing,4956,0,-6.636373,4.535776,10.284599,11.839192,14.216803,14.216803,14.216803


,model,fold,scenario,scope,threshold,false_alert_windows,false_alert_windows_per_hour,packet_true_positives,packet_false_positives,packet_recall,packet_precision,attack_window_recall,attack_windows,maximum_attack_window_logit,in_sample_diagnostic
0,edge_mlp,A,train_dollar_char,frozen_global,10.284599,0,0.000000,0,0,0.000000,NaN,0.000000,3547,6.101065,False
1,edge_mlp,A,train_dollar_char,fold_local_diagnostic,5.959848,4,0.495441,937,16,0.000796,0.983211,0.032140,3547,6.101065,True
2,edge_mlp,A,train_dollar_char,scenario_local_diagnostic,5.959848,4,0.495441,937,16,0.000796,0.983211,0.032140,3547,6.101065,True
3,edge_mlp,A,train_slash_char,frozen_global,10.284599,0,0.000000,0,0,0.000000,NaN,0.000000,2730,6.101064,False
4,edge_mlp,A,train_slash_char,fold_local_diagnostic,5.959848,4,0.434389,4005,16,0.000848,0.996021,0.358974,2730,6.101064,True
5,edge_mlp,A,train_slash_char,scenario_local_diagnostic,5.959848,4,0.434389,4005,16,0.000848,0.996021,0.358974,2730,6.101064,True
6,edge_mlp,A,train_sub_exf,frozen_global,10.284599,0,0.000000,0,0,0.000000,NaN,0.000000,2684,6.101065,False
7,edge_mlp,A,train_sub_exf,fold_local_diagnostic,5.959848,5,0.539245,3527,15,0.006772,0.995765,0.079732,2684,6.101065,True
8,edge_mlp,A,train_sub_exf,scenario_local_diagnostic,5.959848,5,0.539245,3527,15,0.006772,0.995765,0.079732,2684,6.101065,True
9,edge_mlp,B,train_empty_conn,frozen_global,10.284599,0,0.000000,238241,0,0.555375,1.000000,0.043084,5292,14.216803,False


,model,fold,scenario,rank_within_scenario,window_index,window_end_ns,packets,attack_packets,maximum_raw_logit,maximum_attack_raw_logit,...,triggering_source_row_id,triggering_source_sha256,triggering_destination_sha256,triggering_is_arp,triggering_is_ipv4,triggering_is_ipv6,triggering_is_tcp,triggering_is_udp,triggering_is_mqtt,triggering_is_ssh
0,st_gnn,A,train_dollar_char,1,5254,26275009927000,9,0,18.165426,None,...,1945207,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,a9a4b86aface93b17be071d6c3c04928f88ec7b7621afe...,0,1,0,1,0,1,0
1,st_gnn,A,train_dollar_char,2,5253,26270009927000,7,0,18.131094,None,...,1945198,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,80983fa39cac42b2830b969f54bf76b910d5ee8d2e4362...,0,1,0,1,0,1,0
2,st_gnn,A,train_dollar_char,3,5301,26510009927000,5,0,18.120899,None,...,1952972,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,80983fa39cac42b2830b969f54bf76b910d5ee8d2e4362...,0,1,0,1,0,1,0
3,st_gnn,A,train_dollar_char,4,5302,26515009927000,9,0,18.119747,None,...,1952981,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,a9a4b86aface93b17be071d6c3c04928f88ec7b7621afe...,0,1,0,1,0,1,0
4,st_gnn,A,train_dollar_char,5,5278,26395009927000,9,0,18.082348,None,...,1949315,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,a9a4b86aface93b17be071d6c3c04928f88ec7b7621afe...,0,1,0,1,0,1,0
5,st_gnn,A,train_dollar_char,6,5303,26520009927000,12,0,18.080044,None,...,1952986,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,495bf61ea1a9ade0f38c8315419412a856a753530e8ef1...,0,1,0,1,0,1,0
6,st_gnn,A,train_dollar_char,7,5195,25980009927000,12,0,18.063810,None,...,1934503,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,495bf61ea1a9ade0f38c8315419412a856a753530e8ef1...,0,1,0,1,0,1,0
7,st_gnn,A,train_dollar_char,8,5204,26025009927000,17,0,18.051380,None,...,1934635,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,80983fa39cac42b2830b969f54bf76b910d5ee8d2e4362...,0,1,0,1,0,1,0
8,st_gnn,A,train_dollar_char,9,5279,26400009927000,12,0,18.038181,None,...,1949318,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,495bf61ea1a9ade0f38c8315419412a856a753530e8ef1...,0,1,0,1,0,1,0
9,st_gnn,A,train_dollar_char,10,5228,26145009927000,17,0,18.005188,None,...,1939973,c51223413dd23453e9d199c841791a3bfc49a1d89e21c6...,80983fa39cac42b2830b969f54bf76b910d5ee8d2e4362...,0,1,0,1,0,1,0


# cAPTure graph logit tail audit

Status: exploratory diagnostic complete. The frozen global operational threshold remains primary.

## ST-GNN fold score scales

| Fold | Global threshold | Fold diagnostic threshold | Maximum attack logit | Global mean recall | Local mean recall | Local mean FA/h |
|---|---|---|---|---|---|---|
| A | 17.9583 | 17.9583 | 18.8783 | 0.0002 | 0.0002 | 0.9660 |
| B | 17.9583 | 6.3777 | 13.3785 | 0.0000 | 0.8086 | 0.9464 |

## ST-GNN at the frozen global threshold

| Scenario | Fold | Threshold | FA/h | Packet recall | Attack-window recall |
|---|---|---|---|---|---|
| train_dollar_char | A | 17.9583 | 1.4863 | 0.0004 | 0.0248 |
| train_slash_char | A | 17.9583 | 1.4118 | 0.0000 | 0.0000 |
| train_sub_exf | A | 17.9583 | 0.0000 | 0.0000 | 0.0026 |
| train_empty_conn | B | 17.9583 | 0.0000 | 0.0000 | 0.0000 |
| train_qos_mid | B | 17.9583 | 0.0000 | 0.0000 | 0.0000 |

## Diagnostic threshold scopes

`fold_local_diagnostic` and `scenario_local_diagnostic` select and evaluate a threshold on the same OOF predictions. They are optimistic diagnostic counterfactuals, not deployable calibration results. Compare the three scopes in `threshold_scope_metrics.csv`; only `frozen_global` reproduces the primary result.

## Benign upper tail

`top_benign_windows.csv` contains the ten highest-scoring benign windows per scenario and model. `st_gnn_top_window_metadata.csv` joins ST-GNN's selected windows to prepared packets and reports protocol counts, node and pair counts, the triggering packet's protocol flags, and hashes of its endpoints and the most frequent directed pair. Raw MAC addresses are not written.

## Interpretation

Check whether fold or scenario diagnostic thresholds recover recall while satisfying their local budget. Also check whether the benign upper tail overlaps attack scores within each fold. A local recovery suggests score-scale mismatch; persistent low recall suggests within-scenario tail overlap. These mechanisms can coexist. The audit does not establish a cause in the model internals.

No training, checkpoint loading, source overwrite, or held-out scenario access occurred.


## 5. Provenance boundary

In [5]:
if audit_report is not None:
    print("Frozen global threshold unchanged:", audit_report["global_threshold_unchanged"])
    print("Local thresholds are in-sample diagnostics:", audit_report["local_thresholds_are_in_sample_diagnostics"])
    print("Model training performed:", audit_report["model_training_performed"])
    print("Checkpoint loading performed:", audit_report["checkpoint_loading_performed"])
    print("Held-out scenarios accessed:", audit_report["held_out_scenarios_accessed"])
    print("Saved audit:", AUDIT_ARGUMENTS["output_dir"] / "audit_report.json")

Frozen global threshold unchanged: True
Local thresholds are in-sample diagnostics: True
Model training performed: False
Checkpoint loading performed: False
Held-out scenarios accessed: False
Saved audit: /content/drive/MyDrive/capture_gate0/graph_logit_tail_audit_runs/20261002_graph_logit_tail_audit_v1/audit_report.json
